# Kelly Criterion and Bankroll Management on Real Chapter 9 Data

This extra notebook replaces the old synthetic season simulation with the same real 2025-26 Premier League file used in the book rewrite. The goal here is narrower than the main notebook: focus on Kelly sizing, compare it with flat stakes, and show why fractional Kelly is usually safer when probabilities are imperfect.

In [ ]:
import importlib.util
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd().resolve()
ROOT = next(path for path in [cwd, *cwd.parents] if (path / 'Companion-Code').exists())
MODULE_PATH = ROOT / 'Companion-Code' / 'extras' / 'chapter-9' / 'real_data_betting_workflow.py'
spec = importlib.util.spec_from_file_location('chapter9_real', MODULE_PATH)
chapter9_real = importlib.util.module_from_spec(spec)
spec.loader.exec_module(chapter9_real)

matches = chapter9_real.load_matches()
train_df, valid_df = chapter9_real.train_validation_split(matches)
base_model, calibrated_model = chapter9_real.build_calibrated_model(train_df)
summary = chapter9_real.evaluate_model(base_model, calibrated_model, valid_df)
summary_series = pd.Series({
    'Validation Bets': summary['kelly_bets'],
    'Full Kelly Final Bankroll': summary['kelly_final_bankroll'],
    'Full Kelly Profit': summary['kelly_profit'],
    'Full Kelly ROI': summary['kelly_roi'],
    'Flat Stake ROI': summary['flat_value_roi'],
})
summary_series


## Kelly Formula

In [ ]:
def kelly_fraction(probability, odds):
    b = odds - 1
    return max((b * probability - (1 - probability)) / b, 0.0)

examples = pd.DataFrame([
    {'probability': 0.55, 'odds': 2.00},
    {'probability': 0.40, 'odds': 3.20},
    {'probability': 0.28, 'odds': 4.50},
])
examples['kelly_fraction'] = examples.apply(lambda row: kelly_fraction(row['probability'], row['odds']), axis=1)
examples


## Full Kelly vs Fractional Kelly on the Real Validation Set

In [ ]:
classes = summary["classes"]
probs = summary["calibrated_probabilities"]
class_index = {label: i for i, label in enumerate(classes)}

def run_fraction(fraction):
    bankroll = 10_000.0
    starting_bankroll = bankroll
    roi_path = []
    for row_index, row in enumerate(valid_df.itertuples(index=False)):
        best = None
        for outcome, odds_column in [("H", "B365H"), ("D", "B365D"), ("A", "B365A")]:
            odds = getattr(row, odds_column)
            probability = probs[row_index, class_index[outcome]]
            ev = probability * odds - 1
            if best is None or ev > best["ev"]:
                best = {"outcome": outcome, "odds": odds, "probability": probability, "ev": ev}
        if best["ev"] > 0:
            stake = bankroll * kelly_fraction(best["probability"], best["odds"]) * fraction
            bankroll -= stake
            if row.FTR == best["outcome"]:
                bankroll += stake * best["odds"]
        roi_path.append((bankroll - starting_bankroll) / starting_bankroll)
    return roi_path, bankroll

fractions = [1.0, 0.5, 0.25, 0.1, 0.05]
results = {}
for fraction in fractions:
    roi_path, final_bankroll = run_fraction(fraction)
    results[fraction] = {"roi_path": roi_path, "final_bankroll": final_bankroll}

fig, ax = plt.subplots(figsize=(12, 5))
for fraction in fractions:
    ax.plot(valid_df["Kickoff"], results[fraction]["roi_path"], linewidth=2, label=f"{fraction:.0%} Kelly")
ax.axhline(0, color="gray", linestyle="--", linewidth=1)
ax.set_title("Fractional Kelly on the Chapter 9 Validation Block")
ax.set_ylabel("ROI on Starting Bankroll")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

pd.DataFrame({
    'Kelly Fraction': [f'{fraction:.0%}' for fraction in fractions],
    'Final Bankroll': [results[fraction]['final_bankroll'] for fraction in fractions],
    'ROI': [(results[fraction]['final_bankroll'] - 10_000) / 10_000 for fraction in fractions],
})
